In [2]:
# Install required libraries
!pip install -q pandas matplotlib pyarrow scikit-learn lightgbm statsforecast mlforecast neuralforecast


import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, Naive, SeasonalNaive

from mlforecast import MLForecast
from lightgbm import LGBMRegressor

from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS, NHITS

os.makedirs('outputs', exist_ok=True)
os.makedirs('plots', exist_ok=True)

HORIZON = 28
N_WINDOWS = 5
STEP_SIZE = 28
FREQ = 'D'
SEASON_LENGTH = 7
RANDOM_SEED = 1

# Load data
df = pd.read_parquet('sample_hotels.parquet')

# Convert to Nixtla-required format: unique_id, ds, y
df = df.rename(columns={
    'date': 'ds',
    'hotel_id': 'unique_id',
    'demand': 'y'
})

df['ds'] = pd.to_datetime(df['ds'])

# Keep only required forecasting columns
df = df[['unique_id', 'ds', 'y']].copy()
df = df.sort_values(['unique_id', 'ds']).reset_index(drop=True)

print(df.head())
print('Number of hotel series:', df['unique_id'].nunique())
print('Date range:', df['ds'].min(), 'to', df['ds'].max())


def calc_metrics(actual, predicted):
    """Calculate ME, MAE, RMSE, and MAPE. MAPE ignores zero actual values."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    me = np.mean(actual - predicted)
    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))

    nonzero = actual != 0
    if nonzero.sum() == 0:
        mape = np.nan
    else:
        mape = np.mean(np.abs((actual[nonzero] - predicted[nonzero]) / actual[nonzero])) * 100

    return me, mae, rmse, mape


def get_rolling_train_validation(data, fold, h=HORIZON, n_windows=N_WINDOWS):
    """
    Creates one rolling-origin CV fold.
    Fold 1 validates the first of the final 5 windows.
    Fold 5 validates the final 28-day window.
    """
    train_parts = []
    valid_parts = []

    for uid, g in data.groupby('unique_id'):
        g = g.sort_values('ds').reset_index(drop=True)
        n = len(g)
        valid_start = n - (n_windows - fold + 1) * h
        valid_end = valid_start + h

        train_parts.append(g.iloc[:valid_start])
        valid_parts.append(g.iloc[valid_start:valid_end])

    train_fold = pd.concat(train_parts).reset_index(drop=True)
    valid_fold = pd.concat(valid_parts).reset_index(drop=True)

    return train_fold, valid_fold


def make_statsforecast():
    return StatsForecast(
        models=[
            Naive(),
            SeasonalNaive(season_length=SEASON_LENGTH),
            AutoARIMA(season_length=SEASON_LENGTH),
            AutoETS(season_length=SEASON_LENGTH),
        ],
        freq=FREQ,
        n_jobs=-1
    )


def make_mlforecast():
    return MLForecast(
        models=[LGBMRegressor(random_state=RANDOM_SEED, verbose=-1)],
        freq=FREQ,
        lags=[1, 7, 14, 28],
        date_features=['dayofweek', 'month']
    )


def make_neuralforecast():
    # max_steps is intentionally limited so the full 5-fold CV can run in a reasonable time.
    return NeuralForecast(
        models=[
            NBEATS(h=HORIZON, input_size=56, max_steps=300, random_seed=RANDOM_SEED),
            NHITS(h=HORIZON, input_size=56, max_steps=300, random_seed=RANDOM_SEED),
        ],
        freq=FREQ
    )



all_cv_predictions = []

for fold in range(1, N_WINDOWS + 1):
    print(f'Running CV fold {fold} of {N_WINDOWS}...')

    train_fold, valid_fold = get_rolling_train_validation(df, fold)

    # Baseline and statistical models
    sf = make_statsforecast()
    sf.fit(train_fold)
    sf_pred = sf.predict(h=HORIZON).reset_index()

    # Machine learning model
    mlf = make_mlforecast()
    mlf.fit(train_fold)
    ml_pred = mlf.predict(HORIZON).reset_index()

    # Neural models
    nf = make_neuralforecast()
    nf.fit(train_fold)
    nf_pred = nf.predict().reset_index()

    # Merge predictions for this fold
    fold_preds = sf_pred.merge(ml_pred, on=['unique_id', 'ds'], how='left')
    fold_preds = fold_preds.merge(nf_pred, on=['unique_id', 'ds'], how='left')

    # Add actual y values and fold number
    fold_preds = valid_fold.merge(fold_preds, on=['unique_id', 'ds'], how='left')
    fold_preds['fold'] = fold

    all_cv_predictions.append(fold_preds)

cv_predictions = pd.concat(all_cv_predictions).reset_index(drop=True)
cv_predictions.to_csv('outputs/cv_predictions.csv', index=False)

print(cv_predictions.head())


model_cols = [
    'Naive',
    'SeasonalNaive',
    'AutoARIMA',
    'AutoETS',
    'LGBMRegressor',
    'NBEATS',
    'NHITS'
]

eval_rows = []

for (hotel, model), group in cv_predictions.melt(
    id_vars=['unique_id', 'ds', 'y', 'fold'],
    value_vars=model_cols,
    var_name='model',
    value_name='prediction'
).dropna(subset=['prediction']).groupby(['unique_id', 'model']):

    me, mae, rmse, mape = calc_metrics(group['y'], group['prediction'])

    eval_rows.append({
        'unique_id': hotel,
        'model': model,
        'ME': me,
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape
    })

cv_results = pd.DataFrame(eval_rows).sort_values(['unique_id', 'MAE']).reset_index(drop=True)
cv_results.to_csv('outputs/cv_results.csv', index=False)

print(cv_results.head(20))



overall_results = cv_results.groupby('model', as_index=False).agg({
    'ME': 'mean',
    'MAE': 'mean',
    'RMSE': 'mean',
    'MAPE': 'mean'
}).sort_values('MAE')

overall_results.to_csv('outputs/overall_cv_results.csv', index=False)
print(overall_results)



win_tables = []

for metric in ['MAE', 'RMSE', 'MAPE']:
    metric_df = cv_results.dropna(subset=[metric]).copy()
    winners = metric_df.loc[metric_df.groupby('unique_id')[metric].idxmin()]

    counts = winners['model'].value_counts().reset_index()
    counts.columns = ['model', f'{metric}_wins']
    win_tables.append(counts)

model_win_counts = win_tables[0]
for table in win_tables[1:]:
    model_win_counts = model_win_counts.merge(table, on='model', how='outer')

model_win_counts = model_win_counts.fillna(0).sort_values('MAE_wins', ascending=False)
model_win_counts.to_csv('outputs/model_win_counts.csv', index=False)

print(model_win_counts)



# Hold out the final 28 days for a final test comparison
train = df.groupby('unique_id', group_keys=False).apply(lambda x: x.iloc[:-HORIZON]).reset_index(drop=True)
test = df.groupby('unique_id', group_keys=False).apply(lambda x: x.iloc[-HORIZON:]).reset_index(drop=True)

# Fit final models
sf_final = make_statsforecast()
sf_final.fit(train)
sf_forecast = sf_final.predict(h=HORIZON).reset_index()

mlf_final = make_mlforecast()
mlf_final.fit(train)
ml_forecast = mlf_final.predict(HORIZON).reset_index()

nf_final = make_neuralforecast()
nf_final.fit(train)
nf_forecast = nf_final.predict().reset_index()

# Merge final forecasts
forecasts = sf_forecast.merge(ml_forecast, on=['unique_id', 'ds'], how='left')
forecasts = forecasts.merge(nf_forecast, on=['unique_id', 'ds'], how='left')
forecasts.to_csv('outputs/final_forecasts.csv', index=False)

# Save final test set too
test.to_csv('outputs/final_test_actuals.csv', index=False)

print(forecasts.head())



final_eval_rows = []
final_model_cols = [
    'Naive',
    'SeasonalNaive',
    'AutoARIMA',
    'AutoETS',
    'LGBMRegressor',
    'NBEATS',
    'NHITS'
]

for (hotel, model), group in test.merge(forecasts, on=['unique_id', 'ds'], how='inner').melt(
    id_vars=['unique_id', 'ds', 'y'],
    value_vars=final_model_cols,
    var_name='model',
    value_name='prediction'
).dropna(subset=['prediction']).groupby(['unique_id', 'model']):

    me, mae, rmse, mape = calc_metrics(group['y'], group['prediction'])

    final_eval_rows.append({
        'unique_id': hotel,
        'model': model,
        'ME': me,
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape
    })

final_test_results = pd.DataFrame(final_eval_rows).sort_values(['unique_id', 'MAE'])
final_test_results.to_csv('outputs/final_test_results.csv', index=False)

print(final_test_results.head(20))



# Pick the best model for each hotel based on CV MAE
best_by_hotel = cv_results.loc[cv_results.groupby('unique_id')['MAE'].idxmin(), ['unique_id', 'model']]
best_model_map = dict(zip(best_by_hotel['unique_id'], best_by_hotel['model']))

for hotel in df['unique_id'].unique():
    actual = df[df['unique_id'] == hotel]
    pred = forecasts[forecasts['unique_id'] == hotel]
    best_model = best_model_map.get(hotel, overall_results.iloc[0]['model'])

    plt.figure(figsize=(12, 6))
    plt.plot(actual['ds'], actual['y'], label='Actual')

    if best_model in pred.columns:
        plt.plot(pred['ds'], pred[best_model], label=f'{best_model} Forecast')

    plt.title(f'Forecast vs Actual for {hotel} | Best CV Model: {best_model}')
    plt.xlabel('Date')
    plt.ylabel('Room Demand')
    plt.legend()
    plt.tight_layout()

    safe_name = str(hotel).replace('/', '_').replace(' ', '_')
    plt.savefig(f'plots/{safe_name}_forecast.png')
    plt.close()

print('Saved forecast plots for all hotels in the plots folder.')



best_overall = overall_results.iloc[0]['model']
best_mae = overall_results.iloc[0]['MAE']

summary_text = f'''# Main Findings

The strongest overall model based on 5-fold time-series cross-validation was {best_overall}, which had the lowest average MAE across the hotel series. Model performance varied by hotel, which shows why comparing several forecasting approaches was important instead of relying on one method. Neural and machine learning models were useful for capturing more complex demand patterns, while simpler statistical models remained competitive for more stable hotel series. The model win-count table also shows that no single method won every metric for every hotel. Overall, the final forecasting approach supports using model comparison and per-series evaluation for hotel demand planning.
'''

with open('outputs/main_findings_summary.md', 'w') as f:
    f.write(summary_text)

print(summary_text)


  unique_id         ds         y
0   hotel_0 2022-01-01  0.975309
1   hotel_0 2022-01-02  0.493827
2   hotel_0 2022-01-03  0.456790
3   hotel_0 2022-01-04  0.592593
4   hotel_0 2022-01-05  0.530864
Number of hotel series: 18
Date range: 2022-01-01 00:00:00 to 2023-06-30 00:00:00
Running CV fold 1 of 5...


INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.6 M  | train
-------------------------------------------------------
2.6 M     Trainable params
4.8 K     Non-trainable params
2.6 M     Total params
10.231    Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Predicting: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Predicting: |          | 0/? [00:00<?, ?it/s]

   index_x unique_id         ds     Naive  SeasonalNaive  AutoARIMA   AutoETS  \
0        0   hotel_0 2023-06-03  0.691358       0.975309   0.768889  0.665170   
1        1   hotel_0 2023-06-04  0.691358       0.888889   0.652806  0.446198   
2        2   hotel_0 2023-06-05  0.691358       0.271605   0.468714  0.463018   
3        3   hotel_0 2023-06-06  0.691358       0.296296   0.523568  0.497479   
4        4   hotel_0 2023-06-07  0.691358       0.395062   0.586709  0.494638   

   index_y  LGBMRegressor  index    NBEATS     NHITS  
0        0       0.802091      0  0.816857  0.840888  
1        1       0.597872      1  0.556667  0.551615  
2        2       0.606747      2  0.611975  0.565437  
3        3       0.621193      3  0.725183  0.630205  
4        4       0.629974      4  0.718162  0.689020  
    unique_id          model        ME       MAE      RMSE       MAPE
3     hotel_0         NBEATS -0.033336  0.062590  0.073281   9.167830
4     hotel_0          NHITS  0.031008  0.0